In [1]:
import rdflib
from rdflib import Graph, Namespace, URIRef, Literal, BNode
from rdflib.namespace import RDF, RDFS, XSD, OWL
import networkx as nx

In [2]:
nkb = Graph()
nkb.parse('./mappings/NKB_RDF_V3.ttl')

<Graph identifier=N3ae93553ae684110b9b6091b4fd1a50e (<class 'rdflib.graph.Graph'>)>

In [13]:
import rdflib
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from collections import Counter

# Load the RDF
g = rdflib.Graph()
g.parse("./mappings/NKB_RDF_V3.ttl", format="turtle")

# Basic statistics
print(f"Total number of triples: {len(g)}")

# Count entity types
entity_types = []
for s, p, o in g.triples((None, rdflib.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), None)):
    entity_types.append(str(o))

type_counts = Counter(entity_types)
print("\nEntity types and counts:")
for entity_type, count in type_counts.most_common():
    print(f"{entity_type}: {count}")

# Count predicate types
predicates = []
for s, p, o in g:
    predicates.append(str(p))

predicate_counts = Counter(predicates)
print("\nTop 20 predicates:")
for predicate, count in predicate_counts.most_common(20):
    print(f"{predicate}: {count}")

# Count BNodes (nodebags)
bnodes = sum(1 for s in g.subjects() if isinstance(s, rdflib.BNode))
print(f"\nNumber of BNodes (nodebags): {bnodes}")

Total number of triples: 1369675

Entity types and counts:
http://purl.bioontology.org/ontology/npo#NPO_1680: 83233
http://www.bioassayontology.org/bao#BAO_0000179: 24693
http://purl.obolibrary.org/obo/OBI_0000070: 22329
nan: 727
http://purl.bioontology.org/ontology/npo#NPO_199: 374
http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C63495: 302
http://purl.bioontology.org/ontology/npo#NPO_1853: 255
http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C48471: 129
http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C84280: 47
http://purl.enanomapper.org/onto/ENM_0000092: 21
http://purl.bioontology.org/ontology/npo#NPO_174: 16
http://purl.obolibrary.org/obo/ERO_0000833: 10

Top 20 predicates:
http://purl.org/dc/terms/identifier: 284602
http://purl.obolibrary.org/obo/OBI_0002110: 176420
http://www.w3.org/1999/02/22-rdf-syntax-ns#type: 132136
http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C42614: 130957
http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C68553: 112389
http://www.w3.org/199

In [3]:
def convert_rdf_to_knowledge_graph(g):
    """
    Convert an RDF graph to a knowledge graph by extracting the meaning from the RDF structure.
    """
    import rdflib
    import networkx as nx
    from collections import defaultdict
    
    # Create a NetworkX graph
    nx_graph = nx.DiGraph()
    
    # Define entity types with their IRIs
    entity_iris = {
        'material': 'http://purl.bioontology.org/ontology/npo#NPO_199',
        'assay': 'http://purl.obolibrary.org/obo/OBI_0000070',
        'publication': 'http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C48471',
        'medium': 'http://purl.bioontology.org/ontology/npo#NPO_1853',
        'result': 'http://www.bioassayontology.org/bao#BAO_0000179',
        'parameter': 'http://purl.bioontology.org/ontology/npo#NPO_1680',
        'materialfg': 'http://purl.bioontology.org/ontology/npo#NPO_174',
        'additive': 'http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C63495',
        'contam': 'http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C84280',
        'molecularresult': 'http://purl.obolibrary.org/obo/ERO_0000833'
    }
    
    # Map specific predicates to meaningful relationship types
    predicate_to_relation = {
        'http://purl.bioontology.org/ontology/npo#NPO_1853': 'uses_medium',
        'http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C93400': 'uses_material',
        'http://purl.obolibrary.org/obo/OBI_0000070': 'has_assay',
        'http://purl.obolibrary.org/obo/OBI_0002110': 'published_in'
    }
    
    # First step: Add all entities as nodes
    print("Adding entities as nodes...")
    entities = {}
    for entity_type, type_iri in entity_iris.items():
        count = 0
        for s, p, o in g.triples((None, rdflib.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), 
                                 rdflib.URIRef(type_iri))):
            if not isinstance(s, rdflib.BNode):
                str_s = str(s)
                entities[str_s] = {'type': entity_type, 'uri': s, 'properties': {}}
                nx_graph.add_node(str_s, type=entity_type)
                count += 1
        print(f"  Added {count} {entity_type} entities")
    
    print(f"Total: {len(entities)} entities added as nodes")
    
    # Second step: Collect entity properties and identifiers
    print("Collecting entity properties...")
    id_predicate = rdflib.URIRef('http://purl.org/dc/terms/identifier')
    doi_predicate = rdflib.URIRef('http://purl.obolibrary.org/obo/OBI_0002110')
    
    # Create ID and DOI lookups
    id_to_entity = defaultdict(dict)
    doi_to_entity = {}
    
    for entity_uri_str, entity_data in entities.items():
        entity_uri = entity_data['uri']
        entity_type = entity_data['type']
        
        # Find direct properties
        for s, p, o in g.triples((entity_uri, None, None)):
            if isinstance(o, rdflib.Literal):
                if p == id_predicate:
                    id_value = str(o)
                    id_to_entity[entity_type][id_value] = entity_uri_str
                    entity_data['properties']['id'] = id_value
                elif p == doi_predicate:
                    doi_value = str(o)
                    doi_to_entity[doi_value] = entity_uri_str
                    entity_data['properties']['doi'] = doi_value
                else:
                    # Other literal properties
                    prop_name = str(p).split('#')[-1] if '#' in str(p) else str(p).split('/')[-1]
                    entity_data['properties'][prop_name] = str(o)
    
    # Third step: Find entity-to-entity relationships through BNodes
    print("Finding entity relationships through BNodes...")
    relation_counts = defaultdict(int)
    
    for entity_uri_str, entity_data in entities.items():
        entity_uri = entity_data['uri']
        source_type = entity_data['type']
        
        # For each BNode connected to this entity
        for s, p, bnode in g.triples((entity_uri, None, None)):
            if isinstance(bnode, rdflib.BNode):
                # This is a nodebag - extract the relationship type
                relation_type = predicate_to_relation.get(str(p), str(p).split('#')[-1])
                
                # Look for IDs and DOIs in the BNode
                target_entity = None
                for bs, bp, bo in g.triples((bnode, id_predicate, None)):
                    if isinstance(bo, rdflib.Literal):
                        id_value = str(bo)
                        
                        # Determine target type based on the predicate
                        if str(p) == 'http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C93400':
                            target_type = 'material'
                        elif str(p) == 'http://purl.bioontology.org/ontology/npo#NPO_1853':
                            target_type = 'medium'
                        elif str(p) == 'http://purl.obolibrary.org/obo/OBI_0000070':
                            target_type = 'assay'
                        else:
                            continue
                        
                        # Look up the target entity
                        if target_type in id_to_entity and id_value in id_to_entity[target_type]:
                            target_entity = id_to_entity[target_type][id_value]
                            nx_graph.add_edge(entity_uri_str, target_entity, 
                                             relation=relation_type, id=id_value)
                            relation_counts[f"{source_type}_{relation_type}_{target_type}"] += 1
                
                # Check for DOI in the BNode
                for bs, bp, bo in g.triples((bnode, doi_predicate, None)):
                    if isinstance(bo, rdflib.Literal):
                        doi_value = str(bo)
                        if doi_value in doi_to_entity:
                            target_entity = doi_to_entity[doi_value]
                            target_type = entities[target_entity]['type']
                            if target_type == 'publication':
                                nx_graph.add_edge(entity_uri_str, target_entity, 
                                                relation='references_publication', doi=doi_value)
                                relation_counts[f"{source_type}_references_publication"] += 1
    
    # Fourth step: Find direct entity-to-entity relationships
    print("Finding direct entity-to-entity relationships...")
    for entity_uri_str, entity_data in entities.items():
        entity_uri = entity_data['uri']
        source_type = entity_data['type']
        
        # Find direct links to other entities
        for s, p, o in g.triples((entity_uri, None, None)):
            if isinstance(o, rdflib.URIRef) and str(o) in entities:
                target_entity = str(o)
                target_type = entities[target_entity]['type']
                relation_type = predicate_to_relation.get(str(p), str(p).split('#')[-1])
                nx_graph.add_edge(entity_uri_str, target_entity, 
                                 relation=relation_type)
                relation_counts[f"{source_type}_{relation_type}_{target_type}"] += 1
    
    # Fifth step: Handle column-based references (like material_MaterialID)
    print("Finding column-based references...")
    for entity_uri_str, entity_data in entities.items():
        entity_uri = entity_data['uri']
        source_type = entity_data['type']
        
        # Look for column references
        for col_pattern in ['material_MaterialID', 'assay_AssayID', 'medium_MediumID', 'publication_DOI']:
            target_type = col_pattern.split('_')[0]
            for s, p, o in g.triples((entity_uri, rdflib.Literal(col_pattern), None)):
                if isinstance(o, rdflib.Literal):
                    ref_value = str(o)
                    
                    # For DOI special case
                    if col_pattern == 'publication_DOI' and ref_value in doi_to_entity:
                        target_entity = doi_to_entity[ref_value]
                        nx_graph.add_edge(entity_uri_str, target_entity, 
                                         relation='references_publication', doi=ref_value)
                        relation_counts[f"{source_type}_references_publication"] += 1
                    # For ID references
                    elif target_type in id_to_entity and ref_value in id_to_entity[target_type]:
                        target_entity = id_to_entity[target_type][ref_value]
                        nx_graph.add_edge(entity_uri_str, target_entity, 
                                         relation=f"references_{target_type}", id=ref_value)
                        relation_counts[f"{source_type}_references_{target_type}"] += 1
    
    # Print relationship statistics
    print("\nRelationship counts:")
    for rel_type, count in sorted(relation_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"  {rel_type}: {count}")
    
    # Print final graph statistics
    print(f"\nFinal graph: {nx_graph.number_of_nodes()} nodes, {nx_graph.number_of_edges()} edges")
    
    # Count edges by source and target type
    edge_types = defaultdict(int)
    for s, t, data in nx_graph.edges(data=True):
        source_type = nx_graph.nodes[s].get('type', 'unknown')
        target_type = nx_graph.nodes[t].get('type', 'unknown')
        edge_types[(source_type, target_type)] += 1
    
    print("\nEdge counts by source and target type:")
    for (source_type, target_type), count in sorted(edge_types.items(), key=lambda x: x[1], reverse=True):
        print(f"  {source_type} -> {target_type}: {count}")
    
    # Check for isolated nodes
    isolated = list(nx.isolates(nx_graph))
    print(f"\nNumber of isolated nodes: {len(isolated)}")
    
    if isolated:
        isolated_types = defaultdict(int)
        for node in isolated:
            node_type = nx_graph.nodes[node].get('type', 'unknown')
            isolated_types[node_type] += 1
        
        print("Isolated nodes by type:")
        for node_type, count in sorted(isolated_types.items(), key=lambda x: x[1], reverse=True):
            print(f"  {node_type}: {count}")
    
    return nx_graph, entities

In [3]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"MPS (Metal Performance Shaders) available: {torch.backends.mps.is_available()}")
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS device")
else:
    device = torch.device("cpu")
    print("Using CPU device")

PyTorch version: 2.4.1
MPS (Metal Performance Shaders) available: True
Using MPS device


In [4]:
graph, entities = convert_rdf_to_knowledge_graph(nkb)

Adding entities as nodes...
  Added 374 material entities
  Added 22329 assay entities
  Added 129 publication entities
  Added 255 medium entities
  Added 24693 result entities
  Added 83233 parameter entities
  Added 16 materialfg entities
  Added 302 additive entities
  Added 47 contam entities
  Added 10 molecularresult entities
Total: 131388 entities added as nodes
Finding entity relationships through BNodes...
Finding direct entity-to-entity relationships...
Finding column-based references...

Relationship counts:
  parameter_has_assay_assay: 83233
  result_has_assay_assay: 24693
  assay_uses_material_material: 22329
  assay_uses_medium_medium: 22329
  parameter_references_publication: 400
  material_uses_medium_medium: 374
  additive_uses_medium_medium: 302
  result_references_publication: 221
  assay_references_publication: 158
  contam_uses_material_material: 47
  materialfg_uses_material_material: 16
  molecularresult_has_assay_assay: 10

Final graph: 131388 nodes, 154112 edg

In [5]:
def knowledge_graph_link_prediction(nx_graph, entities):
    """
    Perform link prediction on the knowledge graph.
    """
    import networkx as nx
    import numpy as np
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import roc_auc_score, precision_recall_curve, auc
    from sklearn.linear_model import LogisticRegression
    from node2vec import Node2Vec
    
    print("Generating Node2Vec embeddings...")
    # Generate Node2Vec embeddings
    node2vec = Node2Vec(
        nx_graph, 
        dimensions=64,
        walk_length=30,
        num_walks=200,
        workers=4
    )
    
    # Train Node2Vec model
    model = node2vec.fit(window=10, min_count=1)
    
    # Extract embeddings for all nodes
    embeddings = {}
    for node in nx_graph.nodes():
        try:
            embeddings[node] = model.wv[str(node)]
        except KeyError:
            # If node not in vocabulary, use random vector
            embeddings[node] = np.random.rand(64)
    
    # Create edge features
    print("Creating edge features...")
    edge_features = []
    edge_labels = []
    
    # Positive examples (existing edges)
    for u, v in nx_graph.edges():
        # Combine node embeddings (Hadamard product)
        edge_vector = np.multiply(embeddings[u], embeddings[v])
        edge_features.append(edge_vector)
        edge_labels.append(1)
    
    # Negative examples (non-existing edges)
    import random
    random.seed(42)
    nodes = list(nx_graph.nodes())
    negative_samples = 0
    
    # Create an equal number of negative samples
    while negative_samples < len(edge_labels):
        u = random.choice(nodes)
        v = random.choice(nodes)
        if u != v and not nx_graph.has_edge(u, v):
            edge_vector = np.multiply(embeddings[u], embeddings[v])
            edge_features.append(edge_vector)
            edge_labels.append(0)
            negative_samples += 1
    
    # Convert to numpy arrays
    X = np.array(edge_features)
    y = np.array(edge_labels)
    
    # Split into train and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Train logistic regression model
    print("Training link prediction model...")
    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)
    
    # Evaluate model
    y_prob = model.predict_proba(X_test)[:, 1]
    auc_score = roc_auc_score(y_test, y_prob)
    
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    pr_auc = auc(recall, precision)
    
    print(f"Link Prediction Performance:")
    print(f"  ROC AUC: {auc_score:.4f}")
    print(f"  PR AUC: {pr_auc:.4f}")
    
    # Predict missing links between specific entity types
    print("\nPredicting missing links between entity types...")
    
    entity_pairs = [
        ('material', 'assay'),
        ('material', 'medium'),
        ('assay', 'result'),
        ('publication', 'material'),
        ('medium', 'additive')
    ]
    
    potential_links = []
    
    for source_type, target_type in entity_pairs:
        # Get nodes of each type
        source_nodes = [node for node in nx_graph.nodes() 
                        if nx_graph.nodes[node].get('type') == source_type]
        target_nodes = [node for node in nx_graph.nodes() 
                        if nx_graph.nodes[node].get('type') == target_type]
        
        if len(source_nodes) == 0 or len(target_nodes) == 0:
            print(f"  No {source_type} or {target_type} nodes found, skipping")
            continue
        
        print(f"  Evaluating {source_type} -> {target_type}: {len(source_nodes)} x {len(target_nodes)}")
        
        # Sample if there are too many combinations
        max_combinations = 10000
        if len(source_nodes) * len(target_nodes) > max_combinations:
            if len(source_nodes) > len(target_nodes):
                source_nodes = random.sample(source_nodes, max_combinations // len(target_nodes) + 1)
            else:
                target_nodes = random.sample(target_nodes, max_combinations // len(source_nodes) + 1)
        
        # Evaluate potential links
        pair_count = 0
        for source in source_nodes:
            for target in target_nodes:
                if source != target and not nx_graph.has_edge(source, target):
                    edge_vector = np.multiply(embeddings[source], embeddings[target])
                    prob = model.predict_proba([edge_vector])[0, 1]
                    potential_links.append((source, target, prob, source_type, target_type))
                    pair_count += 1
        
        print(f"  Evaluated {pair_count} potential links")
    
    # Sort by probability and return top links
    potential_links.sort(key=lambda x: x[2], reverse=True)
    
    top_k = min(100, len(potential_links))
    print(f"\nTop {top_k} potential missing links:")
    for source, target, prob, s_type, t_type in potential_links[:top_k]:
        source_name = source.split('/')[-1]
        target_name = target.split('/')[-1]
        print(f"  {s_type}({source_name}) -> {t_type}({target_name}): {prob:.4f}")
    
    return model, embeddings, potential_links

In [10]:
nx_graph, entities = convert_rdf_to_knowledge_graph(nkb)

Adding entities as nodes...
  Added 374 material entities
  Added 22329 assay entities
  Added 129 publication entities
  Added 255 medium entities
  Added 24693 result entities
  Added 83233 parameter entities
  Added 16 materialfg entities
  Added 302 additive entities
  Added 47 contam entities
  Added 10 molecularresult entities
Total: 131388 entities added as nodes
Finding entity relationships through BNodes...
Finding direct entity-to-entity relationships...
Finding column-based references...

Relationship counts:
  parameter_has_assay_assay: 83233
  result_has_assay_assay: 24693
  assay_uses_material_material: 22329
  assay_uses_medium_medium: 22329
  parameter_references_publication: 400
  material_uses_medium_medium: 374
  additive_uses_medium_medium: 302
  result_references_publication: 221
  assay_references_publication: 158
  contam_uses_material_material: 47
  materialfg_uses_material_material: 16
  molecularresult_has_assay_assay: 10

Final graph: 131388 nodes, 154112 edg

In [11]:
nx_graph

In [38]:
def precomputed_knowledge_graph_link_prediction_for_directed(nx_graph, entities):
    """
    Perform link prediction with extensive precomputation, handling directed graphs correctly.
    """
    import networkx as nx
    import numpy as np
    import random
    from collections import defaultdict
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import roc_auc_score, precision_recall_curve, auc
    from sklearn.linear_model import LogisticRegression
    import time
    
    start_time = time.time()
    
    print("Precomputing graph metrics...")
    # 1. Precompute node types
    node_types = {}
    type_index = {}
    for i, t in enumerate(set(nx_graph.nodes[n].get('type', 'unknown') for n in nx_graph.nodes())):
        type_index[t] = i
    
    for node in nx_graph.nodes():
        node_type = nx_graph.nodes[node].get('type', 'unknown')
        node_types[node] = type_index.get(node_type, 0)
    
    print(f"  Found {len(type_index)} node types")
    
    # 2. Precompute degrees
    degrees = dict(nx_graph.degree())
    in_degrees = dict(nx_graph.in_degree())
    out_degrees = dict(nx_graph.out_degree())
    max_degree = max(degrees.values()) if degrees else 1
    
    # 3. Precompute neighbor sets for each node (for fast common neighbor calculation)
    print("  Computing neighbor sets...")
    # For directed graphs, we'll use both successors and predecessors as neighbors
    neighbors = {}
    for node in nx_graph.nodes():
        successors = set(nx_graph.successors(node))
        predecessors = set(nx_graph.predecessors(node))
        # Combine both for an undirected view of neighbors
        neighbors[node] = successors.union(predecessors)
    
    # 4. Function to compute common neighbors for directed graphs
    def get_common_neighbors(u, v):
        if u in neighbors and v in neighbors:
            return len(neighbors[u].intersection(neighbors[v]))
        return 0
    
    # 5. Precompute centrality (use degree centrality for speed)
    print("  Computing centrality...")
    centrality = nx.degree_centrality(nx_graph)
    
    # 6. Precompute relation types
    print("  Analyzing relation patterns...")
    relation_types = set()
    for _, _, data in nx_graph.edges(data=True):
        rel_type = data.get('relation', 'unknown')
        relation_types.add(rel_type)
    
    relation_types = list(relation_types)
    rel_type_idx = {rel: i for i, rel in enumerate(relation_types)}
    
    # Store outgoing relation counts for each node
    node_rel_counts = defaultdict(lambda: np.zeros(len(relation_types)))
    for u, v, data in nx_graph.edges(data=True):
        rel_type = data.get('relation', 'unknown')
        if rel_type in rel_type_idx:
            node_rel_counts[u][rel_type_idx[rel_type]] += 1
    
    # 7. Simple node features instead of random walks
    print("  Creating node features...")
    node_features = {}
    for node in nx_graph.nodes():
        # Basic features
        structural_features = np.array([
            node_types[node],  # Type as numeric value
            degrees[node] / max_degree,  # Normalized degree
            in_degrees.get(node, 0) / max_degree,
            out_degrees.get(node, 0) / max_degree,
            centrality.get(node, 0),
            len(neighbors[node]) / max_degree  # Normalized neighborhood size
        ])
        
        # Relation features (normalized)
        rel_features = node_rel_counts[node]
        if len(rel_features) > 0:
            rel_sum = np.sum(rel_features)
            if rel_sum > 0:
                rel_features = rel_features / rel_sum
        
        # Combine
        node_features[node] = np.concatenate([structural_features, rel_features])
    
    feature_dim = len(node_features[list(node_features.keys())[0]])
    print(f"  Created node features with {feature_dim} dimensions")
    print(f"Precomputation completed in {time.time() - start_time:.2f} seconds")
    
    # 8. Define efficient function to compute edge features
    def compute_edge_features(u, v):
        # Calculate common neighbors
        common = get_common_neighbors(u, v)
        
        # Normalize common neighbors
        common_normalized = min(common / 5.0, 1.0)  # Cap at 1.0
        
        # Type similarity
        type_similarity = 1.0 if node_types[u] == node_types[v] else 0.0
        
        # Preferential attachment
        pref_attachment = (degrees[u] * degrees[v]) / (max_degree ** 2)
        
        # Edge basic features
        edge_basic = np.array([type_similarity, common_normalized, pref_attachment])
        
        # Combine with node features
        # Use Hadamard product (element-wise multiplication) for effective feature combination
        node_product = node_features[u] * node_features[v]
        
        # Also add absolute difference between node features (captures asymmetry)
        node_diff = np.abs(node_features[u] - node_features[v])
        
        return np.concatenate([edge_basic, node_product, node_diff])
    
    # 9. Create training data
    print("Creating training data for link prediction...")
    create_start = time.time()
    
    # Define entity type pairs of interest
    entity_type_pairs = [
        ('material', 'assay'),
        ('material', 'medium'),
        ('assay', 'result'),
        ('publication', 'material'),
        ('medium', 'additive')
    ]
    
    # Positive examples: existing edges
    print("  Processing positive examples...")
    edge_features = []
    edge_labels = []
    
    # Process in batches for memory efficiency
    batch_size = 2000
    edges = list(nx_graph.edges())
    
    for i in range(0, len(edges), batch_size):
        batch_edges = edges[i:i+batch_size]
        for u, v in batch_edges:
            edge_vector = compute_edge_features(u, v)
            edge_features.append(edge_vector)
            edge_labels.append(1)
        
        if i % 10000 == 0 and i > 0:
            print(f"    Processed {i}/{len(edges)} edges")
    
    # Negative examples: sample non-edges
    print("  Processing negative examples...")
    random.seed(42)
    negative_count = 0
    max_negatives = len(edge_labels)  # Equal number of positives and negatives
    
    # Preferentially sample from node pairs of different types
    for source_type, target_type in entity_type_pairs:
        if negative_count >= max_negatives:
            break
            
        source_nodes = [node for node in nx_graph.nodes() 
                       if nx_graph.nodes[node].get('type', 'unknown') == source_type]
        target_nodes = [node for node in nx_graph.nodes() 
                        if nx_graph.nodes[node].get('type', 'unknown') == target_type]
        
        if not source_nodes or not target_nodes:
            continue
            
        # Calculate how many negative samples to take from this pair
        samples_per_pair = max(100, max_negatives // len(entity_type_pairs))
        
        # Sample random pairs
        attempts = 0
        while negative_count < max_negatives and attempts < samples_per_pair * 10:
            attempts += 1
            u = random.choice(source_nodes)
            v = random.choice(target_nodes)
            
            if u != v and not nx_graph.has_edge(u, v):
                edge_vector = compute_edge_features(u, v)
                edge_features.append(edge_vector)
                edge_labels.append(0)
                negative_count += 1
                
                if negative_count % 10000 == 0:
                    print(f"    Generated {negative_count}/{max_negatives} negative examples")
    
    # Fill in any remaining negative examples
    if negative_count < max_negatives:
        print(f"    Sampling {max_negatives - negative_count} additional negative examples...")
        nodes = list(nx_graph.nodes())
        
        attempts = 0
        while negative_count < max_negatives and attempts < (max_negatives - negative_count) * 10:
            attempts += 1
            u = random.choice(nodes)
            v = random.choice(nodes)
            
            if u != v and not nx_graph.has_edge(u, v):
                edge_vector = compute_edge_features(u, v)
                edge_features.append(edge_vector)
                edge_labels.append(0)
                negative_count += 1
    
    print(f"Training data created in {time.time() - create_start:.2f} seconds")
    
    # 10. Train link prediction model
    print("Training link prediction model...")
    train_start = time.time()
    
    X = np.array(edge_features)
    y = np.array(edge_labels)
    
    # Split into train and test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Train logistic regression (adjust C for regularization if needed)
    model = LogisticRegression(random_state=42, max_iter=1000, C=1.0)
    model.fit(X_train, y_train)
    
    # Evaluate
    y_prob = model.predict_proba(X_test)[:, 1]
    auc_score = roc_auc_score(y_test, y_prob)
    
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    pr_auc = auc(recall, precision)
    
    print(f"Link Prediction Performance:")
    print(f"  ROC AUC: {auc_score:.4f}")
    print(f"  PR AUC: {pr_auc:.4f}")
    print(f"Model trained in {time.time() - train_start:.2f} seconds")
    
    # 11. Predict missing links
    print("\nPredicting missing links between entity types...")
    predict_start = time.time()
    
    potential_links = []
    
    for source_type, target_type in entity_type_pairs:
        # Get nodes of each type
        source_nodes = [node for node in nx_graph.nodes() 
                        if nx_graph.nodes[node].get('type', 'unknown') == source_type]
        target_nodes = [node for node in nx_graph.nodes() 
                        if nx_graph.nodes[node].get('type', 'unknown') == target_type]
        
        if len(source_nodes) == 0 or len(target_nodes) == 0:
            print(f"  No {source_type} or {target_type} nodes found, skipping")
            continue
        
        print(f"  Evaluating {source_type} -> {target_type}: {len(source_nodes)} x {len(target_nodes)}")
        
        # Sample if there are too many combinations
        max_combinations = 10000
        if len(source_nodes) * len(target_nodes) > max_combinations:
            if len(source_nodes) > len(target_nodes):
                source_nodes = random.sample(source_nodes, max_combinations // len(target_nodes) + 1)
            else:
                target_nodes = random.sample(target_nodes, max_combinations // len(source_nodes) + 1)
        
        # Process in batches
        batch_size = 2000
        candidate_pairs = []
        
        # Create candidate pairs
        for source in source_nodes:
            for target in target_nodes:
                if source != target and not nx_graph.has_edge(source, target):
                    candidate_pairs.append((source, target))
        
        print(f"    Found {len(candidate_pairs)} candidate pairs")
        
        # Process in batches
        for i in range(0, len(candidate_pairs), batch_size):
            batch_pairs = candidate_pairs[i:i+batch_size]
            batch_features = []
            
            for source, target in batch_pairs:
                edge_vector = compute_edge_features(source, target)
                batch_features.append(edge_vector)
            
            # Predict probabilities
            batch_probs = model.predict_proba(np.array(batch_features))[:, 1]
            
            # Store results
            for j, (source, target) in enumerate(batch_pairs):
                prob = batch_probs[j]
                potential_links.append((source, target, prob, source_type, target_type))
            
            if i % 10000 == 0 and i > 0:
                print(f"    Processed {i}/{len(candidate_pairs)} pairs")
    
    print(f"Prediction completed in {time.time() - predict_start:.2f} seconds")
    
    # Sort by probability and return top links
    potential_links.sort(key=lambda x: x[2], reverse=True)
    
    top_k = min(100, len(potential_links))
    print(f"\nTop {top_k} potential missing links:")
    for source, target, prob, s_type, t_type in potential_links[:top_k]:
        source_name = source.split('/')[-1]
        target_name = target.split('/')[-1]
        print(f"  {s_type}({source_name}) -> {t_type}({target_name}): {prob:.4f}")
    
    print(f"Total time: {time.time() - start_time:.2f} seconds")
    
    return model, node_features, potential_links

In [41]:
model, nodes, links = precomputed_knowledge_graph_link_prediction_for_directed(nx_graph, entities)

Precomputing graph metrics...
  Found 10 node types
  Computing neighbor sets...
  Computing centrality...
  Analyzing relation patterns...
  Creating node features...
  Created node features with 10 dimensions
Precomputation completed in 4.88 seconds
Creating training data for link prediction...
  Processing positive examples...
    Processed 10000/154112 edges
    Processed 20000/154112 edges
    Processed 30000/154112 edges
    Processed 40000/154112 edges
    Processed 50000/154112 edges
    Processed 60000/154112 edges
    Processed 70000/154112 edges
    Processed 80000/154112 edges
    Processed 90000/154112 edges
    Processed 100000/154112 edges
    Processed 110000/154112 edges
    Processed 120000/154112 edges
    Processed 130000/154112 edges
    Processed 140000/154112 edges
    Processed 150000/154112 edges
  Processing negative examples...
    Generated 10000/154112 negative examples
    Generated 20000/154112 negative examples
    Generated 30000/154112 negative example

In [51]:
links[40000]

('http://example.org/material/231',
 'http://example.org/medium/112',
 np.float64(0.8808516987930489),
 'material',
 'medium')

In [52]:
def precomputed_knowledge_graph_link_prediction_improved(nx_graph, entities):
    """
    Enhanced knowledge graph link prediction with improved features, model selection,
    and prediction diversity.
    """
    import networkx as nx
    import numpy as np
    import random
    from collections import defaultdict
    from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
    from sklearn.metrics import roc_auc_score, precision_recall_curve, auc
    from sklearn.linear_model import LogisticRegression
    import time
    
    start_time = time.time()
    
    print("Precomputing graph metrics...")
    # 1. Precompute node types
    node_types = {}
    type_index = {}
    for i, t in enumerate(set(nx_graph.nodes[n].get('type', 'unknown') for n in nx_graph.nodes())):
        type_index[t] = i
    
    for node in nx_graph.nodes():
        node_type = nx_graph.nodes[node].get('type', 'unknown')
        node_types[node] = type_index.get(node_type, 0)
    
    print(f"  Found {len(type_index)} node types")
    
    # 2. Precompute degrees
    degrees = dict(nx_graph.degree())
    in_degrees = dict(nx_graph.in_degree())
    out_degrees = dict(nx_graph.out_degree())
    max_degree = max(degrees.values()) if degrees else 1
    max_in_degree = max(in_degrees.values()) if in_degrees else 1
    max_out_degree = max(out_degrees.values()) if out_degrees else 1
    
    # 3. Compute neighbor sets for each node
    print("  Computing neighbor sets...")
    neighbors = {}
    for node in nx_graph.nodes():
        successors = set(nx_graph.successors(node))
        predecessors = set(nx_graph.predecessors(node))
        neighbors[node] = successors.union(predecessors)
    
    # Maximum neighborhood size for normalization
    max_neighbor_size = max(len(n_set) for n_set in neighbors.values()) if neighbors else 1
    
    # 4. Compute centrality and other network metrics
    print("  Computing centrality and other metrics...")
    centrality = nx.degree_centrality(nx_graph)
    
    # Add closeness centrality for the largest component (skip for very large graphs)
    if len(nx_graph) < 10000:
        try:
            # Get largest weakly connected component for directed graph
            largest_cc = max(nx.weakly_connected_components(nx_graph), key=len)
            if len(largest_cc) > 100:  # Only if component is reasonably sized
                subgraph = nx_graph.subgraph(largest_cc)
                closeness = nx.closeness_centrality(subgraph)
            else:
                closeness = {node: 0.0 for node in nx_graph.nodes()}
        except:
            # Fallback if there are issues
            closeness = {node: 0.0 for node in nx_graph.nodes()}
    else:
        closeness = {node: 0.0 for node in nx_graph.nodes()}
    
    # 5. Compute node2vec embeddings if appropriate
    print("  Computing node representation...")
    
    # Simple graph-based representation for efficiency
    # This is a simpler alternative to full node2vec when that's not available
    def compute_simple_node_representation(graph, dim=8):
        """Compute a simple structural embedding for each node."""
        embeddings = {}
        
        for node in graph.nodes():
            # Get local neighborhood structure
            neighborhood = list(neighbors[node])
            neighborhood_size = len(neighborhood)
            
            # Create a feature vector for this node
            embedding = np.zeros(dim)
            
            # Basic structural features for embedding
            if neighborhood_size > 0:
                # Feature 1-2: Avg in/out degree of neighbors
                neighbor_in_degrees = [in_degrees.get(n, 0) for n in neighborhood]
                neighbor_out_degrees = [out_degrees.get(n, 0) for n in neighborhood]
                embedding[0] = sum(neighbor_in_degrees) / neighborhood_size / max_in_degree
                embedding[1] = sum(neighbor_out_degrees) / neighborhood_size / max_out_degree
                
                # Feature 3-4: Fraction of neighbors by types
                neighbor_types = [node_types[n] for n in neighborhood]
                type_counts = defaultdict(int)
                for t in neighbor_types:
                    type_counts[t] += 1
                
                # Use the most common types as features
                top_types = sorted(type_counts.items(), key=lambda x: x[1], reverse=True)[:2]
                if len(top_types) > 0:
                    embedding[2] = top_types[0][0] / len(type_index) if type_index else 0
                    embedding[3] = top_types[0][1] / neighborhood_size
                if len(top_types) > 1:
                    embedding[4] = top_types[1][0] / len(type_index) if type_index else 0
                    embedding[5] = top_types[1][1] / neighborhood_size
            
            # Add some randomness for nodes with similar structures
            embedding[6:] = np.random.normal(0, 0.01, dim - 6)
            
            # Normalize
            norm = np.linalg.norm(embedding)
            if norm > 0:
                embedding = embedding / norm
                
            embeddings[node] = embedding
            
        return embeddings
    
    # Create simplified embeddings
    node_representations = compute_simple_node_representation(nx_graph, dim=8)
    
    # 6. Analyze relation patterns
    print("  Analyzing relation patterns...")
    relation_types = set()
    for _, _, data in nx_graph.edges(data=True):
        rel_type = data.get('relation', 'unknown')
        relation_types.add(rel_type)
    
    relation_types = list(relation_types)
    rel_type_idx = {rel: i for i, rel in enumerate(relation_types)}
    
    # Store outgoing relation counts for each node
    node_rel_counts = defaultdict(lambda: np.zeros(len(relation_types)))
    for u, v, data in nx_graph.edges(data=True):
        rel_type = data.get('relation', 'unknown')
        if rel_type in rel_type_idx:
            node_rel_counts[u][rel_type_idx[rel_type]] += 1
    
    # Normalize relation counts
    for node in node_rel_counts:
        total = sum(node_rel_counts[node])
        if total > 0:
            node_rel_counts[node] = node_rel_counts[node] / total
    
    # 7. Create enhanced node features
    print("  Creating enhanced node features...")
    node_features = {}
    for node in nx_graph.nodes():
        # Basic structural features
        structural_features = np.array([
            node_types[node] / len(type_index) if type_index else 0,  # Normalized type
            degrees[node] / max_degree,                        # Normalized degree
            in_degrees.get(node, 0) / max_in_degree,           # Normalized in-degree
            out_degrees.get(node, 0) / max_out_degree,         # Normalized out-degree
            centrality.get(node, 0),                           # Centrality
            closeness.get(node, 0),                            # Closeness centrality
            len(neighbors[node]) / max_neighbor_size           # Normalized neighborhood size
        ])
        
        # Relation features
        rel_features = node_rel_counts[node]
        
        # Simple representation
        emb_features = node_representations[node]
        
        # Combine all features
        node_features[node] = np.concatenate([
            structural_features,
            rel_features,
            emb_features
        ])
    
    feature_dim = len(node_features[list(node_features.keys())[0]])
    print(f"  Created node features with {feature_dim} dimensions")
    print(f"Precomputation completed in {time.time() - start_time:.2f} seconds")
    
    # 8. Define enhanced edge feature computation
    def compute_enhanced_edge_features(u, v):
        """Compute enhanced edge features for a node pair."""
        # Common neighbors metric
        common = len(neighbors[u].intersection(neighbors[v]))
        common_normalized = min(common / 5.0, 1.0)  # Cap at 1.0
        
        # Advanced structural features
        jaccard = common / (len(neighbors[u]) + len(neighbors[v]) - common) if (len(neighbors[u]) + len(neighbors[v]) - common) > 0 else 0
        adamic_adar = sum(1 / np.log(1 + degrees[w]) for w in neighbors[u].intersection(neighbors[v]) if degrees[w] > 1)
        adamic_adar_norm = min(adamic_adar / 5.0, 1.0)  # Normalize
        
        # Type-based features
        type_similarity = 1.0 if node_types[u] == node_types[v] else 0.0
        
        # Degree-based features
        degree_ratio = min(degrees[u], degrees[v]) / max(degrees[u], degrees[v]) if max(degrees[u], degrees[v]) > 0 else 0
        
        # Combine basic edge features
        edge_basic = np.array([
            common_normalized,
            jaccard,
            adamic_adar_norm,
            type_similarity,
            degree_ratio,
            degrees[u] / max_degree,
            degrees[v] / max_degree
        ])
        
        # Node features interaction
        # Hadamard product (element-wise multiplication)
        node_product = node_features[u] * node_features[v]
        
        # Absolute difference (captures asymmetry)
        node_diff = np.abs(node_features[u] - node_features[v])
        
        return np.concatenate([edge_basic, node_product, node_diff])
    
    # 9. Create training data
    print("Creating training data for link prediction...")
    create_start = time.time()
    
    # Define entity type pairs of interest
    entity_type_pairs = [
        ('material', 'assay'),
        ('material', 'medium'),
        ('assay', 'result'),
        ('publication', 'material'),
        ('medium', 'additive')
    ]
    
    # Process positive examples (existing edges)
    print("  Processing positive examples...")
    edge_features = []
    edge_labels = []
    
    # Process in batches for memory efficiency
    batch_size = 2000
    edges = list(nx_graph.edges())
    
    for i in range(0, len(edges), batch_size):
        batch_edges = edges[i:i+batch_size]
        for u, v in batch_edges:
            edge_vector = compute_enhanced_edge_features(u, v)
            edge_features.append(edge_vector)
            edge_labels.append(1)
        
        if i % 10000 == 0 and i > 0:
            print(f"    Processed {i}/{len(edges)} edges")
    
    # Process negative examples (non-edges)
    print("  Processing negative examples...")
    random.seed(42)
    negative_count = 0
    max_negatives = len(edge_labels)  # Equal number of positives and negatives
    
    # Sample from entity type pairs of interest
    for source_type, target_type in entity_type_pairs:
        if negative_count >= max_negatives:
            break
            
        source_nodes = [node for node in nx_graph.nodes() 
                       if nx_graph.nodes[node].get('type', 'unknown') == source_type]
        target_nodes = [node for node in nx_graph.nodes() 
                        if nx_graph.nodes[node].get('type', 'unknown') == target_type]
        
        if not source_nodes or not target_nodes:
            continue
            
        # Sample pairs
        pairs_to_sample = min(max_negatives // len(entity_type_pairs), 
                               len(source_nodes) * len(target_nodes))
        
        # If there are too many possible pairs, sample efficiently
        if len(source_nodes) * len(target_nodes) > 100000:
            # Sample pairs until we get enough valid ones
            attempts = 0
            while negative_count < max_negatives and attempts < pairs_to_sample * 10:
                attempts += 1
                u = random.choice(source_nodes)
                v = random.choice(target_nodes)
                
                if u != v and not nx_graph.has_edge(u, v):
                    edge_vector = compute_enhanced_edge_features(u, v)
                    edge_features.append(edge_vector)
                    edge_labels.append(0)
                    negative_count += 1
                    
                    if negative_count % 10000 == 0:
                        print(f"    Generated {negative_count}/{max_negatives} negative examples")
        else:
            # For smaller graphs, enumerate all pairs and shuffle
            all_pairs = [(u, v) for u in source_nodes for v in target_nodes 
                         if u != v and not nx_graph.has_edge(u, v)]
            random.shuffle(all_pairs)
            
            for u, v in all_pairs[:pairs_to_sample]:
                edge_vector = compute_enhanced_edge_features(u, v)
                edge_features.append(edge_vector)
                edge_labels.append(0)
                negative_count += 1
                
                if negative_count % 10000 == 0:
                    print(f"    Generated {negative_count}/{max_negatives} negative examples")
                
                if negative_count >= max_negatives:
                    break
    
    # If still need more negative samples, sample randomly
    if negative_count < max_negatives:
        print(f"    Sampling {max_negatives - negative_count} additional negative examples...")
        nodes = list(nx_graph.nodes())
        
        attempts = 0
        while negative_count < max_negatives and attempts < (max_negatives - negative_count) * 10:
            attempts += 1
            u = random.choice(nodes)
            v = random.choice(nodes)
            
            if u != v and not nx_graph.has_edge(u, v):
                edge_vector = compute_enhanced_edge_features(u, v)
                edge_features.append(edge_vector)
                edge_labels.append(0)
                negative_count += 1
    
    print(f"Training data created in {time.time() - create_start:.2f} seconds")
    
    # 10. Train and evaluate model with cross-validation
    print("Training and evaluating link prediction model...")
    train_start = time.time()
    
    X = np.array(edge_features)
    y = np.array(edge_labels)
    
    # Split into train and test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Train with cross-validation to find best parameters
    param_grid = {
        'C': [0.01, 0.1, 1.0],
        'class_weight': [None, 'balanced']
    }
    
    # Use a small subset for CV if data is large
    if len(X_train) > 10000:
        cv_indices = np.random.choice(len(X_train), 10000, replace=False)
        X_cv = X_train[cv_indices]
        y_cv = y_train[cv_indices]
    else:
        X_cv = X_train
        y_cv = y_train
    
    # Cross-validation
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    grid_search = GridSearchCV(
        LogisticRegression(random_state=42, max_iter=1000, solver='liblinear'),
        param_grid=param_grid,
        cv=cv,
        scoring='roc_auc',
        n_jobs=-1
    )
    
    # Find best parameters
    grid_search.fit(X_cv, y_cv)
    print(f"  Best parameters: {grid_search.best_params_}")
    
    # Train final model on all training data with best parameters
    best_model = LogisticRegression(
        random_state=42,
        max_iter=1000,
        solver='liblinear',
        **grid_search.best_params_
    )
    best_model.fit(X_train, y_train)
    
    # Evaluate
    y_prob = best_model.predict_proba(X_test)[:, 1]
    auc_score = roc_auc_score(y_test, y_prob)
    
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    pr_auc = auc(recall, precision)
    
    print(f"Link Prediction Performance:")
    print(f"  ROC AUC: {auc_score:.4f}")
    print(f"  PR AUC: {pr_auc:.4f}")
    print(f"Model trained in {time.time() - train_start:.2f} seconds")
    
    # 11. Predict missing links with diversity
    print("\nPredicting missing links between entity types...")
    predict_start = time.time()
    
    # Track counts for diversity
    source_counts = defaultdict(int)
    target_counts = defaultdict(int)
    diversity_threshold = 5  # Max occurrences of each node
    
    potential_links = []
    
    for source_type, target_type in entity_type_pairs:
        # Get nodes of each type
        source_nodes = [node for node in nx_graph.nodes() 
                        if nx_graph.nodes[node].get('type', 'unknown') == source_type]
        target_nodes = [node for node in nx_graph.nodes() 
                        if nx_graph.nodes[node].get('type', 'unknown') == target_type]
        
        if len(source_nodes) == 0 or len(target_nodes) == 0:
            print(f"  No {source_type} or {target_type} nodes found, skipping")
            continue
        
        print(f"  Evaluating {source_type} -> {target_type}: {len(source_nodes)} x {len(target_nodes)}")
        
        # Sample if there are too many combinations
        max_combinations = 10000
        if len(source_nodes) * len(target_nodes) > max_combinations:
            # Keep all source nodes but sample target nodes
            if len(source_nodes) <= 100:
                targets_per_source = max(1, max_combinations // len(source_nodes))
                sampled_targets = random.sample(target_nodes, min(len(target_nodes), targets_per_source * len(source_nodes)))
                candidate_pairs = [(s, t) for s in source_nodes for t in sampled_targets if s != t and not nx_graph.has_edge(s, t)]
            # Keep all target nodes but sample source nodes
            elif len(target_nodes) <= 100:
                sources_per_target = max(1, max_combinations // len(target_nodes))
                sampled_sources = random.sample(source_nodes, min(len(source_nodes), sources_per_target * len(target_nodes)))
                candidate_pairs = [(s, t) for s in sampled_sources for t in target_nodes if s != t and not nx_graph.has_edge(s, t)]
            # Sample both source and target nodes
            else:
                sampled_sources = random.sample(source_nodes, min(len(source_nodes), int(np.sqrt(max_combinations))))
                sampled_targets = random.sample(target_nodes, min(len(target_nodes), int(np.sqrt(max_combinations))))
                candidate_pairs = [(s, t) for s in sampled_sources for t in sampled_targets if s != t and not nx_graph.has_edge(s, t)]
        else:
            # Use all pairs
            candidate_pairs = [(s, t) for s in source_nodes for t in target_nodes if s != t and not nx_graph.has_edge(s, t)]
        
        # Limit the number of pairs to evaluate
        if len(candidate_pairs) > max_combinations:
            candidate_pairs = random.sample(candidate_pairs, max_combinations)
        
        print(f"    Found {len(candidate_pairs)} candidate pairs")
        
        # Process in batches
        batch_size = 2000
        for i in range(0, len(candidate_pairs), batch_size):
            batch_pairs = candidate_pairs[i:i+batch_size]
            batch_features = []
            
            for source, target in batch_pairs:
                edge_vector = compute_enhanced_edge_features(source, target)
                batch_features.append(edge_vector)
            
            # Predict probabilities
            batch_probs = best_model.predict_proba(np.array(batch_features))[:, 1]
            
            # Store results with diversity check
            for j, (source, target) in enumerate(batch_pairs):
                prob = batch_probs[j]
                
                # Apply a probability threshold and diversity filter
                if prob > 0.5 and source_counts[source] < diversity_threshold and target_counts[target] < diversity_threshold:
                    potential_links.append((source, target, prob, source_type, target_type))
                    source_counts[source] += 1
                    target_counts[target] += 1
            
            if i % 10000 == 0 and i > 0:
                print(f"    Processed {i}/{len(candidate_pairs)} pairs")
        
        print(f"    Evaluated {len(candidate_pairs)} potential links")
    
    print(f"Prediction completed in {time.time() - predict_start:.2f} seconds")
    
    # 12. Apply domain knowledge adjustments
    def domain_knowledge_adjustment(links):
        """Apply domain-specific adjustments to prediction scores."""
        adjusted_links = []
        
        for source, target, prob, s_type, t_type in links:
            adjustment = 0.0
            
            # Example rule: Prefer links between materials and assays
            if s_type == 'material' and t_type == 'assay':
                adjustment += 0.02
                
            # Example rule: Slightly down-weight medium-additive links (they dominated the results)
            if s_type == 'medium' and t_type == 'additive':
                adjustment -= 0.01
                
            # Example rule: Boost links for nodes with few connections
            if degrees[source] < 5 or degrees[target] < 5:
                adjustment += 0.01
            
            # Cap probabilities at 0.999 to avoid perfect 1.0
            adjusted_prob = min(0.999, prob + adjustment)
            adjusted_links.append((source, target, adjusted_prob, s_type, t_type))
        
        return adjusted_links
    
    # Apply adjustments and sort
    enhanced_links = domain_knowledge_adjustment(potential_links)
    enhanced_links.sort(key=lambda x: x[2], reverse=True)
    
    # 13. Display top predictions with diversity
    top_k = min(100, len(enhanced_links))
    print(f"\nTop {top_k} potential missing links:")
    for source, target, prob, s_type, t_type in enhanced_links[:top_k]:
        source_name = source.split('/')[-1]
        target_name = target.split('/')[-1]
        print(f"  {s_type}({source_name}) -> {t_type}({target_name}): {prob:.4f}")
    
    print(f"Total time: {time.time() - start_time:.2f} seconds")
    
    return best_model, node_features, enhanced_links

In [53]:
precomputed_knowledge_graph_link_prediction_improved(nx_graph,entities)

Precomputing graph metrics...
  Found 10 node types
  Computing neighbor sets...
  Computing centrality and other metrics...
  Computing node representation...
  Analyzing relation patterns...
  Creating enhanced node features...
  Created node features with 19 dimensions
Precomputation completed in 2.48 seconds
Creating training data for link prediction...
  Processing positive examples...
    Processed 10000/154112 edges
    Processed 20000/154112 edges
    Processed 30000/154112 edges
    Processed 40000/154112 edges
    Processed 50000/154112 edges
    Processed 60000/154112 edges
    Processed 70000/154112 edges
    Processed 80000/154112 edges
    Processed 90000/154112 edges
    Processed 100000/154112 edges
    Processed 110000/154112 edges
    Processed 120000/154112 edges
    Processed 130000/154112 edges
    Processed 140000/154112 edges
    Processed 150000/154112 edges
  Processing negative examples...
    Generated 10000/154112 negative examples
    Generated 20000/154112


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/pranavsingh/anaconda3/lib/python3.11/site-packages/joblib/externals/loky/backend/popen_loky_posix.py", line 180, in <module>
    exitcode = process_obj._bootstrap()
  File "/Users/pranavsingh/anaconda3/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/pranavsingh/anaconda3/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._a

  Best parameters: {'C': 1.0, 'class_weight': 'balanced'}
Link Prediction Performance:
  ROC AUC: 1.0000
  PR AUC: 1.0000
Model trained in 3.47 seconds

Predicting missing links between entity types...
  Evaluating material -> assay: 374 x 22329
    Found 10000 candidate pairs
    Evaluated 10000 potential links
  Evaluating material -> medium: 374 x 255
    Found 9911 candidate pairs
    Evaluated 9911 potential links
  Evaluating assay -> result: 22329 x 24693
    Found 10000 candidate pairs
    Evaluated 10000 potential links
  Evaluating publication -> material: 129 x 374
    Found 10000 candidate pairs
    Evaluated 10000 potential links
  Evaluating medium -> additive: 255 x 302
    Found 10000 candidate pairs
    Evaluated 10000 potential links
Prediction completed in 0.58 seconds

Top 100 potential missing links:
  material(189) -> medium(249): 0.9990
  material(192) -> medium(99): 0.9990
  material(185) -> medium(249): 0.9990
  material(185) -> medium(99): 0.9990
  material(18

(LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42,
                    solver='liblinear'),
 {'http://example.org/material/0': array([ 4.00000000e-01,  5.35876963e-05,  0.00000000e+00,  3.33333333e-01,
          7.61110308e-06,  0.00000000e+00,  5.35876963e-05,  0.00000000e+00,
          0.00000000e+00,  0.00000000e+00,  1.00000000e+00,  6.15377136e-01,
          0.00000000e+00,  4.92301709e-01,  6.15377136e-01,  0.00000000e+00,
          0.00000000e+00, -1.30951415e-02,  9.46078383e-03]),
  'http://example.org/material/1': array([ 4.00000000e-01,  5.35876963e-05,  0.00000000e+00,  3.33333333e-01,
          7.61110308e-06,  0.00000000e+00,  5.35876963e-05,  0.00000000e+00,
          0.00000000e+00,  0.00000000e+00,  1.00000000e+00,  6.15448728e-01,
          0.00000000e+00,  4.92358982e-01,  6.15448728e-01,  0.00000000e+00,
          0.00000000e+00, -1.54975584e-03, -5.09484657e-03]),
  'http://example.org/material/10': array([4.00000000e-01, 5.35876963e-05, 0

In [64]:
def deep_knowledge_graph_link_prediction(nx_graph, entities):
    """
    Knowledge graph link prediction using deep graph neural networks.
    """
    import networkx as nx
    import numpy as np
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch_geometric.nn import GCNConv, SAGEConv, GATConv
    from torch_geometric.data import Data, HeteroData
    from torch_geometric.transforms import ToUndirected
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import roc_auc_score, precision_recall_curve, auc
    import random
    import time
    from collections import defaultdict
    
    start_time = time.time()
    
    # Check if GPU is available
    device = torch.device('cuda' if torch.cuda.is_available() else 
                         'mps' if torch.backends.mps.is_available() else 
                         'cpu')
    print(f"Using {device} device")
    
    print("Preparing data for GNN...")
    # 1. Extract node types and create mappings
    node_types = {}
    type_to_nodes = defaultdict(list)
    
    for node, attrs in nx_graph.nodes(data=True):
        node_type = attrs.get('type', 'unknown')
        node_types[node] = node_type
        type_to_nodes[node_type].append(node)
    
    print(f"Found {len(type_to_nodes)} node types")
    for type_name, nodes in type_to_nodes.items():
        print(f"  {type_name}: {len(nodes)} nodes")
    
    # 2. Create node type encodings and initial features
    node_to_idx = {}
    features_list = []
    
    for i, node in enumerate(nx_graph.nodes()):
        node_to_idx[node] = i
        
        # Get node's type and degree features
        node_type = node_types[node]
        degree = nx_graph.degree(node)
        in_degree = nx_graph.in_degree(node)
        out_degree = nx_graph.out_degree(node)
        
        # One-hot encode node type (or use embedding lookup later)
        type_idx = list(type_to_nodes.keys()).index(node_type)
        
        # Initial features: [type index, degree, in_degree, out_degree]
        features_list.append([type_idx, degree, in_degree, out_degree])
    
    # 3. Create edge index for PyTorch Geometric
    edge_index = []
    edge_type = []
    edges_list = list(nx_graph.edges(data=True))
    
    for u, v, data in edges_list:
        # Get relation type if available
        rel = data.get('relation', 'unknown')
        
        edge_index.append([node_to_idx[u], node_to_idx[v]])
        edge_type.append(rel)
    
    # Count relation types
    relation_counts = defaultdict(int)
    for rel in edge_type:
        relation_counts[rel] += 1
    
    print(f"Found {len(relation_counts)} relation types")
    for rel, count in sorted(relation_counts.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"  {rel}: {count}")
    
    # 4. Create PyTorch Geometric Data object
    x = torch.tensor(features_list, dtype=torch.float)
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    
    # 5. Create negative samples for training
    print("Creating positive and negative samples...")
    
    # Positive samples: existing edges
    positive_edges = edge_index.t().tolist()
    
    # Create lookup set for fast edge existence check
    edge_set = set([(u, v) for u, v in positive_edges])
    
    # Create negative samples
    negative_edges = []
    node_count = len(node_to_idx)
    
    # Sample negative edges with type constraints
    entity_type_pairs = [
        ('material', 'assay'),
        ('material', 'medium'),
        ('assay', 'result'),
        ('publication', 'material'),
        ('medium', 'additive')
    ]
    
    # Track counts for balanced sampling
    pair_counts = {pair: 0 for pair in entity_type_pairs}
    desired_count = len(positive_edges) // len(entity_type_pairs)
    
    for source_type, target_type in entity_type_pairs:
        # Get nodes of each type
        source_nodes = [node_to_idx[node] for node in type_to_nodes[source_type]]
        target_nodes = [node_to_idx[node] for node in type_to_nodes[target_type]]
        
        if not source_nodes or not target_nodes:
            continue
        
        # Sample pairs until we get enough or run out of combinations
        attempts = 0
        while pair_counts[(source_type, target_type)] < desired_count and attempts < desired_count * 10:
            attempts += 1
            u = random.choice(source_nodes)
            v = random.choice(target_nodes)
            
            if (u != v) and ((u, v) not in edge_set) and ((v, u) not in edge_set):
                negative_edges.append([u, v])
                edge_set.add((u, v))  # Add to set to avoid duplicates
                pair_counts[(source_type, target_type)] += 1
    
    # Fill in any remaining needed negative samples
    while len(negative_edges) < len(positive_edges):
        u = random.randint(0, node_count - 1)
        v = random.randint(0, node_count - 1)
        
        if (u != v) and ((u, v) not in edge_set) and ((v, u) not in edge_set):
            negative_edges.append([u, v])
            edge_set.add((u, v))
    
    print(f"Created {len(positive_edges)} positive and {len(negative_edges)} negative samples")
    
    # 6. Create train/val/test splits
    all_edges = positive_edges + negative_edges
    all_labels = [1] * len(positive_edges) + [0] * len(negative_edges)
    
    # Split into train+val and test
    train_val_edges, test_edges, train_val_labels, test_labels = train_test_split(
        all_edges, all_labels, test_size=0.2, random_state=42, stratify=all_labels
    )
    
    # Split train+val into train and val
    train_edges, val_edges, train_labels, val_labels = train_test_split(
        train_val_edges, train_val_labels, test_size=0.15, random_state=42, stratify=train_val_labels
    )
    
    # 7. Define GNN model with deeper architecture
    class DeepGNN(nn.Module):
        def __init__(self, in_channels, hidden_channels=64, out_channels=32, num_layers=3, dropout=0.3):
            super(DeepGNN, self).__init__()
            
            # First, encode node type as embedding
            num_node_types = len(type_to_nodes)
            self.type_embedding = nn.Embedding(num_node_types, 16)
            
            # Define layers
            self.convs = nn.ModuleList()
            
            # First layer takes raw features plus type embeddings
            self.convs.append(GCNConv(in_channels - 1 + 16, hidden_channels))
            
            # Middle layers
            for _ in range(num_layers - 2):
                self.convs.append(GCNConv(hidden_channels, hidden_channels))
            
            # Last layer
            self.convs.append(GCNConv(hidden_channels, out_channels))
            
            # Dropout for regularization
            self.dropout = dropout
            
            # Edge prediction layers
            self.edge_predictor = nn.Sequential(
                nn.Linear(out_channels * 2, hidden_channels),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_channels, 1)
            )
            
        def encode(self, x, edge_index):
            # Extract type index and process separately
            type_idx = x[:, 0].long()
            type_emb = self.type_embedding(type_idx)
            
            # Replace type index with embedding
            x_features = torch.cat([type_emb, x[:, 1:]], dim=1)
            
            # Apply GNN layers
            for i, conv in enumerate(self.convs):
                x_features = conv(x_features, edge_index)
                if i < len(self.convs) - 1:  # No activation/dropout on last layer
                    x_features = F.relu(x_features)
                    x_features = F.dropout(x_features, p=self.dropout, training=self.training)
            
            return x_features
        
        def decode(self, z, edge_index):
            # For existing edges during training
            row, col = edge_index
            return self.predict(z[row], z[col])
        
        def predict(self, z_src, z_dst):
            # Concatenate source and target embeddings
            z_combined = torch.cat([z_src, z_dst], dim=1)
            # Pass through edge predictor
            return self.edge_predictor(z_combined).squeeze()
        
        def forward(self, x, edge_index, edge_label_index=None):
            # Get node embeddings
            z = self.encode(x, edge_index)
            
            # If edge_label_index is provided, predict those specific edges
            if edge_label_index is not None:
                src, dst = edge_label_index.t()
                return self.predict(z[src], z[dst])
            
            # Otherwise predict the input edges
            return self.decode(z, edge_index)
    
    # 8. Prepare data for training
    # Convert edges to appropriate format
    train_edge_index = torch.tensor(positive_edges, dtype=torch.long).t()
    train_label_index = torch.tensor(train_edges, dtype=torch.long)
    val_label_index = torch.tensor(val_edges, dtype=torch.long)
    test_label_index = torch.tensor(test_edges, dtype=torch.long)
    
    train_y = torch.tensor(train_labels, dtype=torch.float)
    val_y = torch.tensor(val_labels, dtype=torch.float)
    test_y = torch.tensor(test_labels, dtype=torch.float)
    
    # 9. Train the GNN model
    print("Training deep GNN model...")
    
    # Initialize model
    model = DeepGNN(
        in_channels=len(features_list[0]),
        hidden_channels=128,
        out_channels=64,
        num_layers=3,
        dropout=0.3
    ).to(device)
    
    # Define optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)
    
    # Move data to device
    x = x.to(device)
    train_edge_index = train_edge_index.to(device)
    train_label_index = train_label_index.to(device)
    val_label_index = val_label_index.to(device)
    train_y = train_y.to(device)
    val_y = val_y.to(device)
    
    # Training loop
    best_val_auc = 0
    best_epoch = 0
    patience = 10
    patience_counter = 0
    num_epochs = 200
    
    for epoch in range(num_epochs):
        model.train()
        optimizer.zero_grad()
        
        # Forward pass (predict on train edges)
        pred = model(x, train_edge_index, train_label_index)
        
        # Calculate loss
        loss = F.binary_cross_entropy_with_logits(pred, train_y)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Validation
        model.eval()
        with torch.no_grad():
            val_pred = model(x, train_edge_index, val_label_index)
            val_pred_prob = torch.sigmoid(val_pred).cpu().numpy()
            val_auc = roc_auc_score(val_y.cpu().numpy(), val_pred_prob)
        
        # Print progress every 10 epochs
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1:3d}: Loss = {loss.item():.4f}, Val AUC = {val_auc:.4f}")
        
        # Check for improvement
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_epoch = epoch
            patience_counter = 0
            # Save best model state
            best_model_state = {key: value.cpu() for key, value in model.state_dict().items()}
        else:
            patience_counter += 1
        
        # Early stopping
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}. Best validation AUC: {best_val_auc:.4f} at epoch {best_epoch+1}")
            break
    
    # Load best model for evaluation
    model.load_state_dict(best_model_state)
    model = model.to(device)
    
    # 10. Evaluate on test set
    print("\nEvaluating model on test set...")
    model.eval()
    test_label_index = test_label_index.to(device)
    test_y = test_y.to(device)
    
    with torch.no_grad():
        test_pred = model(x, train_edge_index, test_label_index)
        test_pred_prob = torch.sigmoid(test_pred).cpu().numpy()
        test_auc = roc_auc_score(test_y.cpu().numpy(), test_pred_prob)
        
        precision, recall, _ = precision_recall_curve(test_y.cpu().numpy(), test_pred_prob)
        pr_auc = auc(recall, precision)
    
    print(f"Test Performance:")
    print(f"  ROC AUC: {test_auc:.4f}")
    print(f"  PR AUC: {pr_auc:.4f}")
    
    # 11. Generate novel link predictions
    print("\nGenerating novel link predictions...")
    
    # Get node embeddings
    model.eval()
    with torch.no_grad():
        node_embeddings = model.encode(x, train_edge_index).cpu().numpy()
    
    # Store back to original node IDs
    idx_to_node = {idx: node for node, idx in node_to_idx.items()}
    node_embedding_dict = {idx_to_node[i]: emb for i, emb in enumerate(node_embeddings)}
    
    # Predict potential new links
    potential_links = []
    
    # For each type pair, predict potential new links
    for source_type, target_type in entity_type_pairs:
        source_nodes = type_to_nodes[source_type]
        target_nodes = type_to_nodes[target_type]
        
        if not source_nodes or not target_nodes:
            print(f"  No {source_type} or {target_type} nodes found, skipping")
            continue
        
        print(f"  Evaluating {source_type} -> {target_type}: {len(source_nodes)} x {len(target_nodes)}")
        
        # Sample if there are too many combinations
        max_combinations = 10000
        if len(source_nodes) * len(target_nodes) > max_combinations:
            # Keep all nodes of the smaller set
            if len(source_nodes) <= len(target_nodes):
                sampled_sources = source_nodes
                sample_size_targets = max(1, max_combinations // len(source_nodes))
                sampled_targets = random.sample(target_nodes, min(len(target_nodes), sample_size_targets))
            else:
                sampled_targets = target_nodes
                sample_size_sources = max(1, max_combinations // len(target_nodes))
                sampled_sources = random.sample(source_nodes, min(len(source_nodes), sample_size_sources))
                
            candidate_pairs = [(s, t) for s in sampled_sources for t in sampled_targets 
                              if (node_to_idx[s], node_to_idx[t]) not in edge_set
                              and s != t]
        else:
            candidate_pairs = [(s, t) for s in source_nodes for t in target_nodes 
                              if (node_to_idx[s], node_to_idx[t]) not in edge_set
                              and s != t]
        
        print(f"    Found {len(candidate_pairs)} candidate pairs")
        
        # Predict in batches
        batch_size = 1000
        for i in range(0, len(candidate_pairs), batch_size):
            batch_pairs = candidate_pairs[i:i+batch_size]
            
            # Create tensors for batch prediction
            batch_src = [node_to_idx[s] for s, _ in batch_pairs]
            batch_dst = [node_to_idx[t] for _, t in batch_pairs]
            
            # Convert to torch tensors
            batch_edge_index = torch.tensor([batch_src, batch_dst], dtype=torch.long).to(device)
            
            # Predict
            with torch.no_grad():
                batch_z = model.encode(x, train_edge_index)
                batch_src_z = batch_z[batch_edge_index[0]]
                batch_dst_z = batch_z[batch_edge_index[1]]
                batch_pred = model.predict(batch_src_z, batch_dst_z)
                batch_prob = torch.sigmoid(batch_pred).cpu().numpy()
            
            # Store predictions
            for j, (source, target) in enumerate(batch_pairs):
                prob = float(batch_prob[j])
                # Only keep edges with reasonable probability
                if prob > 0.5:
                    potential_links.append((source, target, prob, source_type, target_type))
            
            if i % 5000 == 0 and i > 0:
                print(f"    Processed {i}/{len(candidate_pairs)} pairs")
    
    # 12. Ensure diversity in predictions
    print("\nSelecting diverse predictions...")
    
    # Track counts for diversity
    source_counts = defaultdict(int)
    target_counts = defaultdict(int)
    diversity_threshold = 5  # Max occurrences of each node
    
    # Sort by probability
    potential_links.sort(key=lambda x: x[2], reverse=True)
    
    # Filter for diversity
    diverse_links = []
    for source, target, prob, s_type, t_type in potential_links:
        if source_counts[source] < diversity_threshold and target_counts[target] < diversity_threshold:
            # Cap probability at 0.999 to avoid perfect 1.0 scores
            adjusted_prob = min(0.999, prob)
            diverse_links.append((source, target, adjusted_prob, s_type, t_type))
            source_counts[source] += 1
            target_counts[target] += 1
    
    # 13. Display top predictions
    top_k = min(100, len(diverse_links))
    print(f"\nTop {top_k} potential missing links:")
    for source, target, prob, s_type, t_type in diverse_links[:top_k]:
        source_name = source.split('/')[-1]
        target_name = target.split('/')[-1]
        print(f"  {s_type}({source_name}) -> {t_type}({target_name}): {prob:.4f}")
    
    print(f"Total time: {time.time() - start_time:.2f} seconds")
    
    # Create a map from node names to their embeddings for analysis
    node_embeddings_map = {node: node_embeddings[node_to_idx[node]] for node in nx_graph.nodes()}
    
    return model, node_embeddings_map, diverse_links

In [66]:
mdl, node_embeddings, links = deep_knowledge_graph_link_prediction(nx_graph, entities)

Using mps device
Preparing data for GNN...
Found 10 node types
  material: 374 nodes
  assay: 22329 nodes
  publication: 129 nodes
  medium: 255 nodes
  result: 24693 nodes
  parameter: 83233 nodes
  materialfg: 16 nodes
  additive: 302 nodes
  contam: 47 nodes
  molecularresult: 10 nodes
Found 4 relation types
  has_assay: 107936
  uses_medium: 23005
  uses_material: 22392
  references_publication: 779
Creating positive and negative samples...
Created 154112 positive and 154112 negative samples
Training deep GNN model...
Epoch  10: Loss = 0.3404, Val AUC = 0.9942
Early stopping at epoch 11. Best validation AUC: 0.9961 at epoch 1

Evaluating model on test set...
Test Performance:
  ROC AUC: 0.9960
  PR AUC: 0.9946

Generating novel link predictions...
  Evaluating material -> assay: 374 x 22329
    Found 9685 candidate pairs
    Processed 5000/9685 pairs
  Evaluating material -> medium: 374 x 255
    Found 6683 candidate pairs
    Processed 5000/6683 pairs
  Evaluating assay -> result:

In [67]:
len(links)

784

In [68]:
for i in range(100):
    print(links[i])

('http://example.org/material/276', 'http://example.org/medium/99', 0.9313595294952393, 'material', 'medium')
('http://example.org/material/229', 'http://example.org/medium/99', 0.9313595294952393, 'material', 'medium')
('http://example.org/material/7', 'http://example.org/medium/99', 0.9313574433326721, 'material', 'medium')
('http://example.org/material/169', 'http://example.org/medium/99', 0.9313574433326721, 'material', 'medium')
('http://example.org/material/320', 'http://example.org/medium/99', 0.9313574433326721, 'material', 'medium')
('http://example.org/material/356', 'http://example.org/medium/190', 0.7270485162734985, 'material', 'medium')
('http://example.org/material/84', 'http://example.org/medium/190', 0.7068859338760376, 'material', 'medium')
('http://example.org/material/339', 'http://example.org/medium/190', 0.7068859338760376, 'material', 'medium')
('http://example.org/material/16', 'http://example.org/medium/190', 0.7068858742713928, 'material', 'medium')
('http://e